In [1]:
#importing libraries
import gradio as gr
import pandas as pd
import numpy as np
import faiss
import json
import os
import csv

In [2]:
import shutil
from sentence_transformers import SentenceTransformer

W0713 22:00:30.406000 11196 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [3]:
from llama_cpp import Llama

In [4]:
from sklearn.metrics.pairwise import cosine_similarity
import re
from datetime import datetime
from paddleocr import PaddleOCR
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from textwrap import wrap
from reportlab.lib.utils import ImageReader

In [5]:
# === paths of all files ===
base_dir = r"C:\Users\hp\Downloads\Project folder of Chatter"
faiss_path = os.path.join(base_dir, "faiss_index_PaddleOCR.index")
csv_path = os.path.join(base_dir, "extracted_text_PaddleOCR2/cleaned_text_PaddleOCR2/chunked_text_PaddleOCR.csv")
text_folder = os.path.join(base_dir, "extracted_text_PaddleOCR2/cleaned_text_PaddleOCR2")
logo_path = os.path.join(base_dir, "Logo.png")
mistral_path = r"C:\Users\hp\Downloads\mistral-7b-instruct-v0.1.Q4_K_M.gguf"
CACHE_FILE = "cache.json"
FEEDBACK_FILE = "feedback_log.csv"

In [ ]:
# === Load all components ===

# Load the FAISS index from disk — used for efficient similarity search over embedded text chunks
index = faiss.read_index(faiss_path)

# Load the preprocessed and chunked text data (from OCR'd documents) into a DataFrame
df_chunks = pd.read_csv(csv_path)

# Load the sentence embedding model — used to convert user queries and text chunks into vector form
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Load the local LLM (e.g., Mistral 7B in GGUF format) with context window size set to 2048 tokens
llm = Llama(model_path=mistral_path, n_ctx=2048)

# Initialize the PaddleOCR model with English language and angle classification enabled
ocr_model = PaddleOCR(use_angle_cls=True, lang='en')

In [21]:
# === Session Logs ===
cache = {}
chat_log = []

In [22]:
# === OCR extraction ===

def extract_text_from_image(image_path):
    # Run OCR on image and get results
    ocr_result = ocr_model.ocr(image_path, cls=True)
    # Extract text lines from OCR output and join with newlines
    extracted_text = "\n".join([line[1][0] for block in ocr_result for line in block])
    return extracted_text


In [23]:
# === FAISS retrieval (+ optional OCR chunk) ===

# Perform semantic search over pre-embedded text chunks using a FAISS index
# Optionally, include extra text (e.g., OCR from an uploaded image) as an additional result
def search_top_k(query, k=2, extra_text=None):
    # Convert the query into an embedding vector using the sentence transformer
    embedding = embedding_model.encode([query])
    
    # Search the FAISS index to find the top-k most similar vectors (chunks)
    distances, indices = index.search(np.array(embedding).astype("float32"), k)
    
    # Retrieve the corresponding text chunks (with metadata) from the DataFrame
    faiss_results = df_chunks.iloc[indices[0]][["chunk_id", "filename", "chunk_text"]]
    
    # If additional text (like from OCR) is provided, add it as an extra row in the results
    if extra_text:
        extra_df = pd.DataFrame([{
            "chunk_id": "ocr_chunk",                # Identifier for the dynamic chunk
            "filename": "uploaded_image",           # Label the source as image input
            "chunk_text": extra_text                # Actual OCR-extracted text
        }])
        faiss_results = pd.concat([faiss_results, extra_df], ignore_index=True)

    # Return a DataFrame containing top-k retrieved chunks + optional extra text
    return faiss_results


In [24]:
# === Re-rank retrieved chunks using cosine similarity to refine semantic relevance ===

# Given a set of retrieved chunks, re-rank them by computing cosine similarity between
# the user's question and each chunk to improve semantic accuracy.
def rerank_chunks(chunks_df, question, model):
    # Generate the embedding vector for the input question
    q_vec = model.encode([question])[0]

    # Compute cosine similarity between the question vector and each chunk's embedding
    # Add a new column 'score' to store similarity values
    chunks_df["score"] = chunks_df["chunk_text"].apply(
        lambda text: cosine_similarity([q_vec], [model.encode([text])[0]])[0][0]
    )

    # Sort the chunks in descending order of similarity score and return the top 2
    return chunks_df.sort_values("score", ascending=False).head(2)


In [25]:
# === Best semantic snippet ===

# From a set of retrieved document chunks, identify the single sentence that is most semantically
# similar to the user's question, using cosine similarity on sentence embeddings.
def find_best_semantic_snippet(chunks_df, question, model, max_length=250):
    # Generate embedding for the user's question
    question_vec = model.encode([question])[0]

    # Initialize variables to track the best-matching sentence
    best_snippet, best_file, best_score = "", "", -1

    # Iterate through each chunk (row) in the DataFrame
    for _, row in chunks_df.iterrows():
        # Split chunk text into individual sentences
        sentences = re.split(r'(?<=[.!?]) +', row["chunk_text"])
        
        for sent in sentences:
            sent = sent.strip()
            
            # Skip very short sentences (likely noise)
            if len(sent) < 10:
                continue

            # Compute cosine similarity between question and this sentence
            score = cosine_similarity([question_vec], [model.encode([sent])[0]])[0][0]

            # Update best match if current sentence is more similar
            if score > best_score:
                best_score = score
                best_snippet = sent
                best_file = row["filename"]

    # If no suitable sentence was found, return a fallback (first chunk, truncated)
    if not best_snippet:
        return chunks_df.iloc[0]["filename"], chunks_df.iloc[0]["chunk_text"][:max_length].strip() + "..."

    # Return the best-matching filename and sentence, truncated if too long
    return best_file, best_snippet if len(best_snippet) < max_length else best_snippet[:max_length].strip() + "..."


In [26]:
# === Prompt & answer generation ===
# The prompt follows a structured format to guide the model in generating helpful answers.
def build_prompt(chunks, question, chat_log=None):
    # Concatenate the retrieved text chunks into a single context block
    context = "\n".join(chunks["chunk_text"].tolist())

    # Initialize conversation history string (if available)
    history_text = ""
    if chat_log:
        # Include only the last 2 rounds of Q&A to keep the prompt within context limits
        for i, entry in enumerate(chat_log[-2:], start=1):
            history_text += f"Q{i}: {entry['question']}\nA{i}: {entry['answer']}\n"

    # Compose the full prompt in a structured, instructive format
    prompt = (
        "You are a helpful assistant. Answer the user's question using the provided context.\n\n"
        "### Prior Conversation ###\n" + history_text +
        "### Context from Documents ###\n" + context + "\n\n" +
        "### Question ###\n" + question + "\n\n" +
        "### Answer ###\n"
    )

    # Return the fully constructed prompt for LLM input
    return prompt


In [27]:
def generate_answer(chunks, question, chat_log=None):
    response = llm(build_prompt(chunks, question, chat_log), max_tokens=200, stop=["</s>", "###"])
    return response["choices"][0]["text"].strip()


In [29]:
query = "Who is Marieve Herington?"

# Step 1: Retrieve relevant chunks from FAISS index
relevant_chunks = search_top_k(query, k=3)

# Step 2: Generate answer using those chunks
answer = generate_answer(relevant_chunks, query, chat_log)

print("Answer:", answer)


llama_perf_context_print:        load time =  226221.35 ms
llama_perf_context_print: prompt eval time =  226209.99 ms /   977 tokens (  231.54 ms per token,     4.32 tokens per second)
llama_perf_context_print:        eval time =   58950.58 ms /    94 runs   (  627.13 ms per token,     1.59 tokens per second)
llama_perf_context_print:       total time =  285265.02 ms /  1071 tokens


Answer: Marieve Herington is a Canadian actress and jazz singer who has appeared in recurring roles on television shows such as How I Met Your Mother, Good Luck Charlie, and Ever After High. She also provides the voice of Tilly Green in the Disney Channel show Big City Greens. Herington has been fronting her own jazz ensembles since the age of 16, and has released several albums under her own label, Maribelle Records.
